<a href="https://colab.research.google.com/github/DarshiniMahesh/AI-Practice/blob/main/Flower_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [ ]:
!kaggle datasets download -d pramithks/flowers-recognition-dataset

Dataset URL: https://www.kaggle.com/datasets/pramithks/flowers-recognition-dataset
License(s): unknown
100% 290M/290M [00:03<00:00, 101MB/s]



In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('/content/flowers-recognition-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Training data
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

training_set = train_datagen.flow_from_directory(
    '/content/Flower_recognition/training_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

Found 4317 images belonging to 5 classes.


In [ ]:
# Testing data
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory(
    '/content/Flower_recognition/test_set',
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical'
)

Found 1020 images belonging to 5 classes.


In [ ]:
#Model Training (with Validation)
cnn = tf.keras.models.Sequential()

cnn.add(tf.keras.layers.Conv2D(filters=64, kernel_size=3, activation='relu', input_shape=[64,64,3]))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

cnn.add(tf.keras.layers.Conv2D(filters=64, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

cnn.add(tf.keras.layers.Dropout(0.5))

cnn.add(tf.keras.layers.Flatten())

cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

cnn.add(tf.keras.layers.Dense(units=5, activation='softmax'))

cnn.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

cnn.fit(x=training_set, validation_data=test_set, epochs=30)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 65s 464ms/step - accuracy: 0.4360 - loss: 1.3281 - val_accuracy: 0.5618 - val_loss: 1.0606
Epoch 2/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 60s 447ms/step - accuracy: 0.5733 - loss: 1.0674 - val_accuracy: 0.6265 - val_loss: 0.9593
Epoch 3/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 58s 431ms/step - accuracy: 0.6152 - loss: 0.9643 - val_accuracy: 0.6853 - val_loss: 0.8316
Epoch 4/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 80s 417ms/step - accuracy: 0.6588 - loss: 0.8894 - val_accuracy: 0.6853 - val_loss: 0.8096
Epoch 5/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 56s 414ms/step - accuracy: 0.6776 - loss: 0.8392 - val_accuracy: 0.6794 - val_loss: 0.8283
Epoch 6/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 54s 403ms/step - accuracy: 0.6891 - loss: 0.8075 - val_accuracy: 0.7549 - val_loss: 0.6561
Epoch 7/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 55s 408ms/step - accuracy: 0.7137 - loss: 0.7595 - val_accuracy: 0.7804 - val_loss: 0.5929
Epoch 8/30
135/135 ━━━━━━━━━━━━━━━━━━━━ 55s 409ms/step - accuracy: 0.7320 - loss: 0

In [ ]:
# Save model so you never retrain again
cnn.save('flower_model.h5')
print("Model saved!")

Model saved!


In [ ]:
#Label Encoding / Class Mapping Retrieval
training_set.class_indices

{'daisy': 0, 'dandelion': 1, 'rose': 2, 'sunflower': 3, 'tulip': 4}

In [ ]:
# Step 1: Import Image Preprocessing Module
from tensorflow.keras.preprocessing import image


# Step 2: Image Loading & Resizing (Inference Input Preparation)
test_image = image.load_img('/content/rose.jpg', target_size=(64,64))


# Step 3: Image Conversion to Numerical Array
test_image = image.img_to_array(test_image)


# Step 4: Adding Batch Dimension (Model Input Formatting)
test_image = np.expand_dims(test_image, axis=0)


# Step 5: Model Prediction (Inference Stage)
result = cnn.predict(test_image)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step


In [ ]:
# Load model (use this if you're in a fresh Colab session)
# from tensorflow.keras.models import load_model
# cnn = load_model('flower_model.h5')

# Class names in the same order as training_set.class_indices
class_names = ['Daisy', 'Dandelion', 'Rose', 'Sunflower', 'Tulip']

# Get prediction scores for all 5 flowers
result = cnn.predict(test_image)
scores = result[0]  # [0.02, 0.05, 0.91, 0.01, 0.01]

# Find the winner — highest score
predicted_index = np.argmax(scores)
predicted_flower = class_names[predicted_index]
confidence = scores[predicted_index] * 100

print(f"Prediction  : {predicted_flower}")
print(f"Confidence  : {confidence:.2f}%")
print()
print("All scores:")
for name, score in zip(class_names, scores):
    bar = "█" * int(score * 30)
    print(f"  {name:<12} {score*100:5.1f}%  {bar}")

In [ ]:
# Install gradio (only needed once per Colab session)
!pip install gradio -q

import gradio as gr
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load your saved model
cnn = load_model('flower_model.h5')
class_names = ['Daisy', 'Dandelion', 'Rose', 'Sunflower', 'Tulip']

def predict_flower(img):
    # Resize and preprocess exactly like training
    img_resized = img.resize((64, 64))
    img_array  = image.img_to_array(img_resized)
    img_array  = img_array / 255.0               # same rescale as training
    img_array  = np.expand_dims(img_array, axis=0)

    # Predict
    scores = cnn.predict(img_array)[0]

    # Return as dict — Gradio draws the bar chart automatically
    return {class_names[i]: float(scores[i]) for i in range(5)}

# Build the interface
app = gr.Interface(
    fn=predict_flower,
    inputs=gr.Image(type="pil", label="Upload a flower photo"),
    outputs=gr.Label(num_top_classes=5, label="Prediction"),
    title="Flower Recognition",
    description="Upload any flower image — model will identify if it's a Daisy, Dandelion, Rose, Sunflower or Tulip",
    examples=[],
    theme=gr.themes.Soft()
)

app.launch(share=True, debug=True)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'flower_model.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)